# Deterministic Workflow Execution: Pure Python Nodes & State Schemas
**Production Reference Implementation (Senior AI Engineer Standard)**

## Architectural Scope
This notebook implements an LLM-free deterministic workflow using **LangGraph**.
It isolates core graph mechanics to validate:
1. State schema definitions using `TypedDict` (`graph_info: str`).
2. Functional node contracts accepting state objects and returning atomic partial updates (`deltas`).
3. Type-safe conditional edge routing using `typing.Literal`.
4. End-to-end graph compilation with entry point (`START`) and terminal sinks (`END`).
5. Traversal and state transition verification across discrete supersteps.
6. Programmatic validation assertions ensuring deterministic state progression.


In [1]:
import random
from typing import Dict, Any, Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

print("LangGraph core graph components and typing modules imported successfully.")


LangGraph core graph components and typing modules imported successfully.


## 1. State Schema Engineering with TypedDict
The state schema serves as the canonical shared contract across all nodes.
We use `TypedDict` to enforce static type hints without runtime serialization overhead.


In [2]:
class PlayState(TypedDict):
    """Canonical state schema holding shared workflow information."""
    graph_info: str

print("PlayState schema defined with key [graph_info: str].")


PlayState schema defined with key [graph_info: str].


## 2. Pure Functional Node Contracts
Each node is a Python function accepting the state schema as its primary argument
and returning a dictionary containing the delta update for the state.


In [3]:
def start_play_node(state: PlayState) -> Dict[str, Any]:
    """Initial node initializing intent and appending planning information."""
    current_info = state.get("graph_info", "")
    print(f"[Node: start_play] Received graph_info: '{current_info}'")
    updated_info = f"{current_info} I am planning to play".strip()
    return {"graph_info": updated_info}

def cricket_node(state: PlayState) -> Dict[str, Any]:
    """Target branch node when cricket activity is selected."""
    current_info = state.get("graph_info", "")
    print(f"[Node: cricket] Appending cricket activity.")
    return {"graph_info": f"{current_info} cricket"}

def badminton_node(state: PlayState) -> Dict[str, Any]:
    """Target branch node when badminton activity is selected."""
    current_info = state.get("graph_info", "")
    print(f"[Node: badminton] Appending badminton activity.")
    return {"graph_info": f"{current_info} badminton"}

print("Node functions defined: start_play_node, cricket_node, badminton_node.")


Node functions defined: start_play_node, cricket_node, badminton_node.


## 3. Type-Safe Conditional Routing
The conditional decision function inspects state or probabilistic triggers
and returns a string constrained by `typing.Literal` matching valid destination nodes.


In [4]:
def decide_game_branch(state: PlayState) -> Literal["cricket", "badminton"]:
    """Conditional edge routing function returning node destination."""
    threshold = random.random()
    decision: Literal["cricket", "badminton"] = "cricket" if threshold > 0.5 else "badminton"
    print(f"[Conditional Edge: decide_game] Sampled threshold {threshold:.3f} -> Routing to '{decision}'")
    return decision

print("Conditional routing function defined with return type Literal['cricket', 'badminton'].")


Conditional routing function defined with return type Literal['cricket', 'badminton'].


## 4. Graph Assembly and Compilation
We instantiate `StateGraph` with the `PlayState` schema, register nodes,
bind direct and conditional edges, and compile the executable runnable.


In [5]:
# Instantiate graph builder
builder = StateGraph(PlayState)

# Register computational nodes
builder.add_node("start_play", start_play_node)
builder.add_node("cricket", cricket_node)
builder.add_node("badminton", badminton_node)

# Add deterministic entry transition
builder.add_edge(START, "start_play")

# Add conditional routing edge from start_play
builder.add_conditional_edges(
    "start_play",
    decide_game_branch,
    {
        "cricket": "cricket",
        "badminton": "badminton"
    }
)

# Add terminal sink transitions to END
builder.add_edge("cricket", END)
builder.add_edge("badminton", END)

# Compile into executable runnable graph
workflow_app = builder.compile()

print("Graph assembled and compiled successfully.")


Graph assembled and compiled successfully.


## 5. Workflow Execution and Stream Traversal
We execute the workflow using stream traversal to observe superstep state transformations in real time.


In [6]:
initial_payload: PlayState = {"graph_info": "Workflow Started:"}

print("Initiating graph superstep stream traversal...")
print("=" * 60)

executed_nodes = []
for step_idx, step_output in enumerate(workflow_app.stream(initial_payload)):
    for node_label, state_delta in step_output.items():
        print(f"Superstep {step_idx + 1} -> Node Executed: [{node_label}]")
        print(f"  State Delta: {state_delta}")
        executed_nodes.append(node_label)
    print("-" * 60)

final_result = workflow_app.invoke(initial_payload)
print("Final Consolidated State Execution Result:")
print("  graph_info:", final_result["graph_info"])
print("  Traversed Node Sequence:", executed_nodes)


Initiating graph superstep stream traversal...
[Node: start_play] Received graph_info: 'Workflow Started:'
[Conditional Edge: decide_game] Sampled threshold 0.652 -> Routing to 'cricket'
Superstep 1 -> Node Executed: [start_play]
  State Delta: {'graph_info': 'Workflow Started: I am planning to play'}
------------------------------------------------------------
[Node: cricket] Appending cricket activity.
Superstep 2 -> Node Executed: [cricket]
  State Delta: {'graph_info': 'Workflow Started: I am planning to play cricket'}
------------------------------------------------------------
[Node: start_play] Received graph_info: 'Workflow Started:'
[Conditional Edge: decide_game] Sampled threshold 0.637 -> Routing to 'cricket'
[Node: cricket] Appending cricket activity.
Final Consolidated State Execution Result:
  graph_info: Workflow Started: I am planning to play cricket
  Traversed Node Sequence: ['start_play', 'cricket']


## 6. Programmatic Invariant Assertions
Defensive validation confirming deterministic state mutation and routing compliance.


In [7]:
# 1. Verify entry node executed
assert "start_play" in executed_nodes, "Assertion Failed: start_play node must be executed."

# 2. Verify mutually exclusive branching
has_cricket = "cricket" in executed_nodes
has_badminton = "badminton" in executed_nodes
assert has_cricket != has_badminton, "Assertion Failed: Exactly one branch (cricket or badminton) must execute."

# 3. Verify state string concatenation integrity
final_text = final_result["graph_info"]
assert "Workflow Started:" in final_text, "Assertion Failed: Initial state missing."
assert "I am planning to play" in final_text, "Assertion Failed: start_play mutation missing."
assert ("cricket" in final_text or "badminton" in final_text), "Assertion Failed: Destination sport missing."

print("All deterministic workflow assertions passed successfully.")


All deterministic workflow assertions passed successfully.
